# M0 B2 — Intégrer une IA sur étagère (variante scikit-learn)
---

## Etape 1 - Prise en main de l'environnement
---
2 fichiers données sources
   - breast-cancer-train.csv
   - breast-cancer-infer-6a102cdec94b5435031773.csv

Échantillons → ce sont les observations, en d'autres termes : les lignes du tableau

Features (caractéristiques) → ce sont les variables explicatives,  en d'autres termes : les colonnes (hors target)

Classes (labels) → ce sont les valeurs possibles à prédire



In [47]:
import numpy as np #On importe numpy en lui donnat l'alias 'np' par covention
print("NumPy version :", np.__version__)

NumPy version : 2.2.6


In [48]:
data = np.loadtxt("breast-cancer-train.csv", delimiter=",", skiprows=1)

X = data[:, :-1]   # toutes les colonnes sauf la dernière
y = data[:, -1]    # dernière colonne (Classe - target)

# Calcul des informations du fichier
n_samples = X.shape[0]   # nombre de lignes
n_features = X.shape[1]  # nombre de colonnes
n_classes = len(np.unique(y))  # valeurs uniques

# Affichage des résultats
print("=== Dataset Info ===")
print(f"Nombre d'échantillons (samples) : {n_samples}")
print(f"Nombre de features             : {n_features}")
print(f"Nombre de classes              : {n_classes}")

# Liste des classes
print("Classes :", np.unique(y))



=== Dataset Info ===
Nombre d'échantillons (samples) : 559
Nombre de features             : 30
Nombre de classes              : 2
Classes : [0. 1.]


## Étape 2 — Entraînement et sérialisation du modèle
---

### Import des librairies

In [49]:
import joblib
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import pandas as pd

### Chargement des données

In [50]:
df = pd.read_csv("breast-cancer-train.csv") # Import des données du fichier CSV dans un dataframe pandas

In [51]:
df.head(3) # Affichage des 3 premières lignes du fichier

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,16.02,23.24,102.7,797.8,0.08206,0.06669,0.03299,0.03323,0.1528,0.05697,...,33.88,123.8,1150.0,0.1181,0.1551,0.1459,0.09975,0.2948,0.08452,0
1,15.78,17.89,103.6,781.0,0.09710,0.12920,0.09954,0.06606,0.1842,0.06082,...,27.28,136.5,1299.0,0.1396,0.5609,0.3965,0.18100,0.3792,0.10480,0
2,19.17,24.80,132.4,1123.0,0.09740,0.24580,0.20650,0.11180,0.2397,0.07800,...,29.94,151.7,1332.0,0.1037,0.3903,0.3639,0.17670,0.3176,0.10230,0


### Séparation features / target

In [52]:
# 
X = df.drop('target', axis=1) # création d'un dataframe contenant uniquement les features
y = df['target'] # Création d'un dataframe contenant uniquement la target

In [53]:
print("Dataset Breast Cancer")
print(f"Nombre d'échantillons : {X.shape[0]}") # Calcul du nombre de lignes (échantillons) du dataframe
print(f"Nombre de features    : {X.shape[1]}")  # Calcul du nombre de colonnes (features) du dataframe

Dataset Breast Cancer
Nombre d'échantillons : 559
Nombre de features    : 30


### Séparation jeu de train / test
Cette partie consiste à faire 2 groupe d'échantillons : 
- 1 pour l'entrainement
- 1 pour le test

L'objectif étant d'entrainer et tester le modèe sur des données différentes,  afin déviter l'overfitting. Le modèle peut ainsi être entrainé sur une partie des données puis être testé et évalué sur l'autre partie.
Dans notre cas, on effectue une division aléatoire des données en 2 ensemble : 80 % pour l'entrainement, 20 % pour le test. 

In [54]:
# Utilisation de la fonction train_test_split de sklearn
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # taille de l'échantillon de test : 20%
    random_state=42,    # permet de garantir la reproductibilité du split
    stratify=y          # la classe étant déséquilibrée (357 1 pour 202 0), il garantit que la répartition des classes est respectée entre train et test
)

In [55]:
print(f"Train : {X_train.shape[0]} échantillons") # Nombre d'échantillons dans le set d'entrainement
print(f"Test  : {X_test.shape[0]} échantillons")  # Nombre d'échantillons dans le set de test

Train : 447 échantillons
Test  : 112 échantillons


### Mise à l'échelle des features (standardisation)
L'objectif de cette étape est de ramener toutes les variables du jeu de données à une échelle comparable afin d’éviter qu’une variable ayant des valeurs plus élevées ne domine le modèle. Elle garanti une contribution équilibrée des différentes features. 
La classe `StandardScaler` de la librairie sklearn permet de normaliser les features.
Note : on garde les 2 jeu de données séparés, entrainement et test.

In [56]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# X_train_scaled = X_train
# X_test_scaled = X_test

### Entrainement
La classe `LogisticRegression` permet de créer un modèle de classification (0 ou 1). Le paramètre `max_iter=1000` fixe le nombre maximal d’itérations pour l’apprentissage, tandis que `random_state=42` garantit la reproductibilité des résultats. La méthode `fit()` entraîne le modèle en apprenant les coefficients qui permettent de prédire la variable cible à partir des données d’entraînement.

In [57]:
model = LogisticRegression(max_iter=1000, random_state=42) # création du model
model.fit(X_train_scaled, y_train) # entrainement du modèle

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


### Evaluation
#### Prédiction et performance

In [58]:
y_pred   = model.predict(X_test_scaled) # utilise le modèle entraîné (fit) pour prédire la classe des données de test
accuracy = model.score(X_test_scaled, y_test) # Evalue la performance du modèle en comparant le résultat des prédictions et la classe de test

In [59]:
print(f"\nAccuracy sur le test set : {accuracy:.4f}") # formattage accuracy avec 4 décimales
print("\nRapport de classification :")
print(classification_report(y_test, y_pred)) # rapport détaillé de performance du modèle


Accuracy sur le test set : 0.9732

Rapport de classification :
              precision    recall  f1-score   support

           0       0.97      0.95      0.96        40
           1       0.97      0.99      0.98        72

    accuracy                           0.97       112
   macro avg       0.97      0.97      0.97       112
weighted avg       0.97      0.97      0.97       112



**Note :** La méthode `predict()` permet de générer les classes prédites à partir des données de test. La méthode `score()` calcule ensuite la précision du modèle en comparant les prédictions aux valeurs réelles. L’accuracy correspond au pourcentage de prédictions correctes sur l’ensemble des données de test. Cependant, l'accuracy seule peut être trompeuse. L'accuracy peut être affinée grace au rapport de performance `classification_report`.

#### Matrice de confusion

In [60]:
print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred))

Matrice de confusion :
[[38  2]
 [ 1 71]]


La matrice de confusion indique :
   - 38 true negative
   - 71 true negative
   - 2 False positive
   - 1 False negative

#### Sauvegarde

Cette partie consiste à sérialiser le modèle. Elle consiste à persister les données et la définition du modèle dans des fichiers afin de faire tourner le modèle ultérieurement sans le ré-entrainer.
La fonction joblib.dump() permet de sauvegarder le modèle et le scaler sur disque. On sauvegarde séparément le modèle et le scaler afin d’appliquer les mêmes transformations aux nouvelles données avant de faire des prédictions cohérentes.

In [61]:
SCALER_PATH = "scaler.joblib"
MODEL_PATH  = "model.joblib"
 
# Le scaler et le modèle sont sauvegardés séparément.
# Les deux sont nécessaires pour faire une prédiction correcte.
joblib.dump(scaler, SCALER_PATH) # génère le fichier scaler.joblib
joblib.dump(model,  MODEL_PATH) # génère le fichier model.joblib

['model.joblib']

#### Rechargement des modèles

Ce code permet de recharger le modèle et le scaler précédemment sauvegardés, puis de tester leur bon fonctionnement sur un exemple de données. Les données sont d’abord normalisées à l’aide du scaler, puis passées au modèle pour obtenir une prédiction et les probabilités associées.

In [62]:
# Vérification : rechargement et inférence sur un exemple

loaded_scaler = joblib.load(SCALER_PATH)
loaded_model  = joblib.load(MODEL_PATH)
 
sample_raw    = X_test[0:1]          # données brutes : récupère la première ligne du jeu de test
# print(sample_raw)
sample_scaled = loaded_scaler.transform(sample_raw)  # normalisation de l'échantillon
prediction    = loaded_model.predict(sample_scaled)  # prédiction
proba         = loaded_model.predict_proba(sample_scaled) # indique le niveau de confiance du modèle

print(proba)

[[9.61863984e-05 9.99903814e-01]]


In [64]:
prediction, y_test[0:1]

(array([1]),
 310    1
 Name: target, dtype: int64)

### Questions
   - Quelle est la différence entre `precision` et `recall` ? Laquelle privilégier dans un contexte médical ?

      - `precision` mesure la fiabilité des prédictions positives : parmis les positifs prédits, combien sont corrects.
     
      - `recall` mesure la capacité à ne rien rater : parmis les vrais positifs, combien ont été trouvés.

      => La  `precision` privilégie la qualité en évitant les faux positifs alors que `recall` privilégie la quantité en évitant les faux négatifs. Dans un contexte médical, on aura tendance à privilégier le `recall`, l'objectif étant de ne rater aucun positif.   
    
   - Pourquoi utiliser `stratify=y` dans le `train_test_split` ?

       - la classe étant déséquilibrée (357 1 pour 202 0), le `stratify=y` garantit que la répartition des classes est respectée entre le jeu d'entrainement et le jeu de test. Il évite un biais qui pourrait créé du à un mauvaise répartition.

   - Que contient exactement le fichier `model.joblib` ?

        - Ce fichier de format binaire contient la définition du modèle, c'est à dire le modèle lui-même (LogisticRegression), les paramètres appris pendant l'entrainement, des métadonnées, et autres objets python (dictionnaire de classes, fonctions, etc..). 


## Etape 3 - Exposition via FastAPI
---

**Question de réflexion** : Pourquoi le modèle est-il chargé une seule fois au démarrage du serveur, et non à chaque requête ?

Le chargement du modèle au démarrage du serveur constitue une optimisation : le modèle est chargé en mémoire au démarrage du serveur et chaque requête appelle donc le modèle directement en mémoire.
    
Un chargement du modèle à chaque requête génèrerait de la latence (lecture disque du.joblib + montée en mémoire)

En d'autres termes, charger un modèle à chaque requête consitue un goulot d'etranglement et conduit a un effondrement des performances en cas de multiples requêtes simultaneés.

## Étape 4 — Interface utilisateur Streamlit
---

Question de réflexion : L'interface affiche les probabilités `probability_malignant` et `probability_benign` — leur somme vaut-elle toujours 1 ? Pourquoi ?

Le modèle réalise une classification binaire (0 ou 1). Les 2 classes sont excusives et exhaustives : une tumeur est soit begnine, soit maligne. Les 2 probabilités sont complémentaires, mathématiquement : **P(malignant)+P(benign)=1**

## Bonus - Amélioration du modèle
---

### Bonus 1 — Comparer plusieurs algorithmes

Remplacer la LogisticRegression par un RandomForestClassifier ou un SVC. Comparer les métriques.